# 🥈 Silver Layer — Crypto Data Transformation

**Proyecto:** CoinMarketCap Analytics Platform  
**Capa:** Silver  
**Fuente:** Tabla Bronze `crypto_project.bronze_crypto_markets_raw`  
**Responsabilidad:** Transformar el JSON crudo en columnas limpias, tipadas y listas para análisis.

**Regla Silver:** los datos se limpian, se estructuran y se validan, pero todavía no se calculan KPIs finales de negocio.

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType
)

from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    current_timestamp,
    when
)

In [0]:
spark.sql("USE SCHEMA crypto_project")

In [0]:
bronze_df = spark.table("crypto_project.bronze_crypto_markets_raw")

display(bronze_df)

In [0]:
crypto_json_schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", DoubleType(), True),
    StructField("market_cap_rank", LongType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percentage_24h", DoubleType(), True),
    StructField("market_cap_change_24h", DoubleType(), True),
    StructField("market_cap_change_percentage_24h", DoubleType(), True),
    StructField("circulating_supply", DoubleType(), True),
    StructField("total_supply", DoubleType(), True),
    StructField("max_supply", DoubleType(), True),
    StructField("ath", DoubleType(), True),
    StructField("ath_change_percentage", DoubleType(), True),
    StructField("ath_date", StringType(), True),
    StructField("atl", DoubleType(), True),
    StructField("atl_change_percentage", DoubleType(), True),
    StructField("atl_date", StringType(), True),
    StructField("last_updated", StringType(), True)
])

In [0]:
parsed_df = bronze_df.withColumn(
    "parsed_json",
    from_json(col("raw_json"), crypto_json_schema)
)

display(parsed_df)

In [0]:
silver_df = parsed_df.select(
    col("parsed_json.id").alias("coin_id"),
    col("parsed_json.symbol").alias("symbol"),
    col("parsed_json.name").alias("coin_name"),
    col("parsed_json.current_price").alias("current_price_usd"),
    col("parsed_json.market_cap").alias("market_cap_usd"),
    col("parsed_json.market_cap_rank").alias("market_cap_rank"),
    col("parsed_json.total_volume").alias("total_volume_usd"),
    col("parsed_json.high_24h").alias("high_24h_usd"),
    col("parsed_json.low_24h").alias("low_24h_usd"),
    col("parsed_json.price_change_24h").alias("price_change_24h_usd"),
    col("parsed_json.price_change_percentage_24h").alias("price_change_percentage_24h"),
    col("parsed_json.market_cap_change_24h").alias("market_cap_change_24h_usd"),
    col("parsed_json.market_cap_change_percentage_24h").alias("market_cap_change_percentage_24h"),
    col("parsed_json.circulating_supply").alias("circulating_supply"),
    col("parsed_json.total_supply").alias("total_supply"),
    col("parsed_json.max_supply").alias("max_supply"),
    col("parsed_json.ath").alias("all_time_high_usd"),
    col("parsed_json.ath_change_percentage").alias("all_time_high_change_percentage"),
    to_timestamp(col("parsed_json.ath_date")).alias("all_time_high_date"),
    col("parsed_json.atl").alias("all_time_low_usd"),
    col("parsed_json.atl_change_percentage").alias("all_time_low_change_percentage"),
    to_timestamp(col("parsed_json.atl_date")).alias("all_time_low_date"),
    to_timestamp(col("parsed_json.last_updated")).alias("last_updated_ts"),
    col("source_system"),
    col("api_endpoint"),
    col("ingestion_ts"),
    current_timestamp().alias("silver_processed_ts")
)

display(silver_df)

In [0]:
silver_df.select(
    "coin_id",
    "symbol",
    "coin_name",
    "current_price_usd",
    "market_cap_usd",
    "market_cap_rank"
).show(truncate=False)

In [0]:
silver_df.select(
    [
        when(col(c).isNull(), 1).otherwise(0).alias(c)
        for c in [
            "coin_id",
            "symbol",
            "coin_name",
            "current_price_usd",
            "market_cap_usd",
            "market_cap_rank"
        ]
    ]
).groupBy().sum().show()

In [0]:
silver_table = "crypto_project.silver_crypto_markets_clean"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print(f"Tabla Silver creada correctamente: {silver_table}")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM crypto_project.silver_crypto_markets_clean
    ORDER BY market_cap_rank ASC
    """)
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_registros
FROM crypto_project.silver_crypto_markets_clean
""").show()

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY crypto_project.silver_crypto_markets_clean
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)